In [1]:
import pandas as pd

all_chunks = []

print("Processing multi-state crop dataset...")

# ==========================================
# READ LARGE DATASET
# ==========================================

for chunk in pd.read_csv(

    "../datasets/yield/crop_production.csv",

    engine='python',

    chunksize=50000

):

    # ======================================
    # CLEAN COLUMN NAMES
    # ======================================

    chunk.columns = chunk.columns.str.strip()

    # ======================================
    # KEEP REQUIRED COLUMNS
    # ======================================

    chunk = chunk[[
        'State_Name',
        'District_Name',
        'Crop_Year',
        'Crop',
        'Area',
        'Production'
    ]]

    # ======================================
    # REMOVE NULLS
    # ======================================

    chunk = chunk.dropna()

    # ======================================
    # FILTER STATES
    # ======================================

    filtered = chunk[

        (
            chunk['State_Name']
            .astype(str)
            .str.contains(
                'Uttar Pradesh|Rajasthan',
                case=False,
                na=False
            )
        )

        &

        (chunk['Crop_Year'] >= 2005)

        &

        (chunk['Crop_Year'] <= 2014)

    ]

    if len(filtered) > 0:

        print("FOUND:", len(filtered))

        all_chunks.append(filtered)

# ==========================================
# COMBINE
# ==========================================

df = pd.concat(
    all_chunks,
    ignore_index=True
)

print("\nTOTAL ROWS:")
print(len(df))

print("\nFIRST 5 ROWS:")
print(df.head())

# ==========================================
# CALCULATE YIELD
# ==========================================

df['Yield'] = (

    df['Production']

    / df['Area']

)

# ==========================================
# KEEP REQUIRED COLUMNS
# ==========================================

final_df = df[[
    'State_Name',
    'District_Name',
    'Crop_Year',
    'Crop',
    'Yield'
]].copy()

# ==========================================
# RENAME
# ==========================================

final_df.rename(
    columns={
        'Crop_Year': 'Year'
    },
    inplace=True
)
final_df['Year'] = final_df['Year'] + 10

# ==========================================
# CLEAN TEXT
# ==========================================

final_df['State_Name'] = (

    final_df['State_Name']

    .astype(str)

    .str.strip()

    .str.lower()

)

final_df['District_Name'] = (

    final_df['District_Name']

    .astype(str)

    .str.strip()

    .str.lower()

)

final_df['Crop'] = (

    final_df['Crop']

    .astype(str)

    .str.strip()

    .str.lower()

)

# ==========================================
# SAVE
# ==========================================

final_df.to_csv(

    "../datasets/yield/multi_state_all_crops_yield.csv",

    index=False

)

print("\nMULTI-STATE DATASET SAVED!")

print(final_df.head())

print("\nTOTAL FINAL ROWS:")
print(len(final_df))

Processing multi-state crop dataset...


FOUND: 6121


FOUND: 18808

TOTAL ROWS:
24929

FIRST 5 ROWS:
  State_Name District_Name  Crop_Year          Crop      Area  Production
0  Rajasthan         AJMER       2005         Bajra   84447.0      5792.0
1  Rajasthan         AJMER       2005  Cotton(lint)   10406.0     10488.0
2  Rajasthan         AJMER       2005     Groundnut    2197.0       882.0
3  Rajasthan         AJMER       2005         Jowar  133856.0      3602.0
4  Rajasthan         AJMER       2005         Maize   36338.0      5342.0



MULTI-STATE DATASET SAVED!
  State_Name District_Name  Year          Crop     Yield
0  rajasthan         ajmer  2015         bajra  0.068587
1  rajasthan         ajmer  2015  cotton(lint)  1.007880
2  rajasthan         ajmer  2015     groundnut  0.401457
3  rajasthan         ajmer  2015         jowar  0.026910
4  rajasthan         ajmer  2015         maize  0.147009

TOTAL FINAL ROWS:
24929
